# Embedding Techniques

converting texts into vectors

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

OPENAI EMBEDDINGS

In [4]:
from langchain_openai import OpenAIEmbeddings

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings 

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7fe1ae543970>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7fe1aee51030>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [6]:
text = "This is a tutorial for OPENAI embeddings"
query_result = embeddings.embed_query(text)
print(query_result)

[-0.004823852796107531, 0.03563227877020836, -0.018049709498882294, -0.03433467447757721, 0.027664974331855774, -0.006799458991736174, 0.009576337411999702, 0.05481090769171715, -0.008356587029993534, 0.006150655914098024, -0.015571280382573605, 0.0011256737634539604, -0.008084089495241642, -0.012748985551297665, 0.014247721992433071, 0.01682995818555355, 0.022915733978152275, 0.04624670371413231, -0.019684692844748497, -0.018529823049902916, 0.0003982030611950904, 0.005563488695770502, -0.046428367495536804, -0.038694631308317184, 0.024965953081846237, -0.02601701393723488, 0.013235588558018208, 0.03674822300672531, -0.012392143718898296, 0.05133332312107086, 0.009816395118832588, -0.021501343697309494, 0.011360546573996544, 0.0017047307919710875, -0.023123351857066154, 0.016323892399668694, 0.0490235798060894, 0.02392786741256714, -0.03672226890921593, 0.018880177289247513, 0.02994876354932785, -4.724099562736228e-05, -0.0006427208427339792, -0.024784287437796593, -0.0534613952040672

In [8]:
len(query_result) #takes input text and converts it into 3072 vectors (3072 is dimension by default)

3072

if we want to set our own choice dimension (eg 1024): 

embeddings = OpenAIEmbeddings(model="text-embedding-3-large",dimensions=1024)

entire process for one document:

In [17]:
embeddings_1024= OpenAIEmbeddings(model="text-embedding-3-large",dimensions=1024)


In [21]:
# loading
from langchain_community.document_loaders import TextLoader
loader = TextLoader("speech_text.txt")
docs = loader.load()

In [22]:
docs

[Document(metadata={'source': 'speech_text.txt'}, page_content='Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as \'Bebo\')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal granddaughter of actor Hari Shivdasani, and great-granddaughter of filmmaker Prithviraj Kapoor. The actors Rishi Kapoor and Neetu Kapoor are her uncle and aunt respectively, and actor Ranbir Kapoor is also her cousin.\n\nKareena and Karisma Kapoor with their mother Babita\nPictured with mother Babita (left) and sister Karisma. In an interview with journalist Vir Sanghvi, Kapoor stated that growing up with the two of them helped her become strong and independent.[5]\nAccording to Kapoor, her name "Kareena" was derived from the book Anna Karenina, which her mother read while she was pregnant with her.[6] Kapoor is of Punjabi

In [23]:
#splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_docs = splitter.split_documents(docs)


In [24]:
final_docs[0]

Document(metadata={'source': 'speech_text.txt'}, page_content="Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as 'Bebo')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal granddaughter of actor Hari Shivdasani, and great-granddaughter of filmmaker Prithviraj Kapoor. The actors Rishi Kapoor and Neetu Kapoor are her uncle and aunt respectively, and actor Ranbir")

In [26]:
#Vector Embedding and Vector StoreDB (combining both these steps)
from langchain_community.vectorstores import Chroma
db = Chroma.from_documents(final_docs,embeddings)


In [27]:
#Retrieve the results from query vectorstore db.

query = "Kareena and Karisma Kapoor with their mother Babita"
retrieved_result = db.similarity_search(query)
print(retrieved_result)

[Document(metadata={'source': 'speech_text.txt'}, page_content='Kareena and Karisma Kapoor with their mother Babita\nPictured with mother Babita (left) and sister Karisma. In an interview with journalist Vir Sanghvi, Kapoor stated that growing up with the two of them helped her become strong and independent.[5]'), Document(metadata={'source': 'speech_text.txt'}, page_content='Kareena and Karisma Kapoor with their mother Babita\nPictured with mother Babita (left) and sister Karisma. In an interview with journalist Vir Sanghvi, Kapoor stated that growing up with the two of them helped her become strong and independent.[5]'), Document(metadata={'source': 'speech_text.txt'}, page_content="Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as 'Bebo')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal gr

OLLAMA EMBEDDIGNS

In [28]:
from langchain_community.embeddings import OllamaEmbeddings

In [32]:
embedding = (
    OllamaEmbeddings(model="gemma2:2b")
)

In [33]:
embedding

OllamaEmbeddings(base_url='http://localhost:11434', model='gemma2:2b', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [34]:
r1 = embedding.embed_documents([
    "alpha is the first letter of greek alphabet",
    "beta is the second letter of greek alphabet"
])

In [36]:
len(r1[0])

2304

In [38]:
r1[1]

[-0.8567498326301575,
 -0.05211279168725014,
 -0.9441795349121094,
 0.6151219606399536,
 0.09004312753677368,
 1.7354189157485962,
 0.38254591822624207,
 0.35239937901496887,
 -0.7313967943191528,
 1.0795553922653198,
 -2.1048593521118164,
 -0.5532231330871582,
 -0.11866508424282074,
 -1.289765477180481,
 0.6736483573913574,
 -0.39403876662254333,
 -0.7760729789733887,
 -1.6150339841842651,
 -5.287760257720947,
 -0.8804775476455688,
 1.7037549018859863,
 0.20367315411567688,
 -1.8292572498321533,
 -0.34313663840293884,
 2.675204038619995,
 -1.2038582563400269,
 1.4414609670639038,
 1.245986819267273,
 -0.08741223067045212,
 0.9776169657707214,
 -1.1264235973358154,
 0.20766907930374146,
 -1.6737875938415527,
 -0.8089733123779297,
 0.8141016364097595,
 1.4716341495513916,
 -1.071722149848938,
 0.7339737415313721,
 0.8443179726600647,
 -2.2355551719665527,
 -0.45158183574676514,
 2.096344232559204,
 -0.846807599067688,
 0.08588618785142899,
 -0.582177460193634,
 -1.7177915573120117,
 -0.

In [37]:
embedding.embed_query("what is the second letter of greek alphabet ")

[-0.2178451418876648,
 0.2671235203742981,
 -3.44301700592041,
 0.6782328486442566,
 1.5287147760391235,
 0.7748650312423706,
 0.7225764989852905,
 0.005201439838856459,
 0.25254473090171814,
 1.8525162935256958,
 -2.6874780654907227,
 0.25303542613983154,
 -0.4882340729236603,
 -1.8715529441833496,
 0.611272931098938,
 0.729413628578186,
 -9.839827537536621,
 0.5724306702613831,
 -2.9854629039764404,
 0.04014560207724571,
 2.02939772605896,
 1.3862675428390503,
 -0.358630895614624,
 1.8773843050003052,
 1.7650048732757568,
 -1.3838132619857788,
 2.2653586864471436,
 -2.8306691646575928,
 -0.29229429364204407,
 -0.2535172998905182,
 -0.7836580872535706,
 1.1828707456588745,
 -3.0661184787750244,
 -1.0718724727630615,
 -0.23649778962135315,
 3.209996461868286,
 -0.9493222236633301,
 -0.5481927394866943,
 -0.07090344280004501,
 0.2748720645904541,
 -0.9839035868644714,
 1.9161773920059204,
 0.25099533796310425,
 -0.20171675086021423,
 -0.5148114562034607,
 -0.8742213845252991,
 0.4631305

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['HUGGINGFACE_API_KEY'] = os.getenv("HUGGINGFACE_API_KEY")

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

/Users/user/Desktop/genai/venv/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [53]:
import os
os.environ["HF_HOME"] = "/Users/user/Desktop/genai/hfcache"


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
text = "this is a test document"
result = embeddings.embed_query(text)
result

[-0.04820789024233818,
 0.11789610981941223,
 -0.03746984899044037,
 0.05662048980593681,
 0.015501733869314194,
 -0.03674931824207306,
 -0.05957154929637909,
 0.05720917880535126,
 -0.02075633779168129,
 0.05708468705415726,
 0.07765137404203415,
 0.018936678767204285,
 0.0006362574058584869,
 -0.00040408241329714656,
 -0.06529416143894196,
 -0.028550028800964355,
 -0.011813527904450893,
 -0.04569054767489433,
 -0.007525638211518526,
 0.08929043263196945,
 0.053103744983673096,
 0.06305597722530365,
 -0.00455248961225152,
 0.0003609198611229658,
 0.008460155688226223,
 0.030092934146523476,
 -0.06308866292238235,
 0.03805890679359436,
 0.0815841555595398,
 -0.05816447362303734,
 0.03205762058496475,
 0.06851986050605774,
 0.07797116786241531,
 0.0341692753136158,
 0.06476924568414688,
 0.004228681791573763,
 0.07276925444602966,
 0.0025179029908031225,
 0.03840569034218788,
 0.03519797697663307,
 -0.017656540498137474,
 -0.11470243334770203,
 0.0094097089022398,
 0.03478091210126877,


In [6]:
len(result)

384

In [7]:
doc_result = embeddings.embed_documents([text,"this isnt text document"])
print(doc_result)

[[-0.04820787534117699, 0.11789615452289581, -0.03746984899044037, 0.056620463728904724, 0.015501736663281918, -0.03674929216504097, -0.05957154557108879, 0.05720913037657738, -0.02075633592903614, 0.057084646075963974, 0.07765141129493713, 0.018936708569526672, 0.0006362562417052686, -0.0004040762723889202, -0.06529418379068375, -0.028550054877996445, -0.011813536286354065, -0.04569051414728165, -0.007525628432631493, 0.08929039537906647, 0.05310376361012459, 0.06305600702762604, -0.004552487749606371, 0.00036090690991841257, 0.008460129611194134, 0.030092893168330193, -0.06308868527412415, 0.03805890306830406, 0.08158417791128159, -0.058164454996585846, 0.03205762431025505, 0.06851985305547714, 0.07797116041183472, 0.03416929021477699, 0.06476924568414688, 0.004228699952363968, 0.07276921719312668, 0.0025179176591336727, 0.03840572014451027, 0.035197991877794266, -0.017656538635492325, -0.11470239609479904, 0.009409739635884762, 0.03478090092539787, 0.02008695900440216, 0.02663722448

In [11]:
print(len(doc_result[1]))

384
